# Suite REF — References: one format for objects, relation entries, frames and parameters

A reference designates what a request or a response points at ([CAP-130: references to objects, entries and frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-130.md#decided)):
`References.OfObject`, an object, by its path (a singleton, an owned object or a
schema object), never a counted one, which nothing names (object identity 40 and 55 in [CAP-118: singletons, owned and counted objects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-118.md#decided));
`References.OfEntry`, a relation entry, by its relation's path and all its links and properties; `References.OfFrame`,
a frame, each symbol's object and each variable's value, and what it witnesses, objects' property values
(`References.OfProperties`) and relation entries, each by its objects' references, and the errors annotated on it
(`References.OfError`, [CAP-309: errors annotated on frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-309)). `References.of(store, x)`
gives one, a value of the store's, written and read as any value is. A reference is resolved within a transaction, as
every read of a store's data is ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md)): a map whose symbol is given an object's reference
binds what it designates, and gives no frame where the store holds none; a frame's reference is resolved by a map
whose symbols are given the references it binds, and an entry's is matched by a pattern over the objects it links. `References.OfParameter`, a parameter, holds its name and
nothing else; a binding in the scope that encloses it resolves it, not a store. A parameter reference may stand in the place of anything with a schema
([CAP-312: a parameter reference in place of anything with a schema](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-312.md#decided)): it is read through `Visitors.OfAny.as_parameter`, every other
visitor reads its place as empty, and the plain form writes it marked with its schema where the reader could take it for
a value of the schema expected.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, YAML, Builders as B, Changes, Comparison as C, Paths, Plain, Proxies, Reachable, \
    References, Transactions, Validators
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import raises


def crm(txn):
    """crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by key; crm.Note, counted,
    about a person."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation, "Book": S.OfObject,
                  "Note": S.OfObject, "About": S.OfRelation})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]).properties({
                "since": S.OfInt}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
            ]).adjacencies({"trio": lambda t: t.relation([S.OfRelation(lambda r: r.links([  # no name of its own
                lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))]).me("a")}),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({
                "key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}).adjacencies({
                "about": lambda t: t.relation([_.About]).me("note")}),
            lambda _: _.About.name("crm.About").links([lambda t: t.name("note"), lambda t: t.name("person")]),
        ])]))])


store = B.Store()
store.transaction(crm)
Person, Book, Note = (store.registered(name) for name in ("crm.Person", "crm.Book", "crm.Note"))
frames = store.transaction(lambda t: t.uid("people").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"book": Book, "ada": Person, "bob": Person, "cy": Person}).spec([
        lambda _: _.book.contacts({"ada": lambda e: e.contact(_.ada), "bob": lambda e: e.contact(_.bob),
                                   "cy": lambda e: e.contact(_.cy)}),
        lambda _: _.ada.name("Ada").knows([lambda e: e.known(_.bob).since(2020), lambda e: e.known(_.cy).since(2021)]),
        lambda _: _.bob.name("Bob"),
        lambda _: _.cy.name("Cy"),
    ])]))]))
book, ada, bob, cy = (frames[0][name] for name in ("book", "ada", "bob", "cy"))  # each its reference: a frame holds no visitor


def contact(key):
    """The contact the book holds by `key`, within the transaction open: found again by navigating from the singleton."""
    return store.singleton("crm.TheBook").contacts({"key": key})


def notes(t):
    """A note about each person, upserted with the key `notes/1` for the frame that reaches it."""
    return t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"contact": Person})])
        .then([lambda n: n.upsert(lambda u: u
        .key("notes/1").exists([lambda p: p.symbols({"contact": Person, "note": Note}).spec([
            lambda _: _.note.text("hi").about([_.contact])])]))]))])


def plain(reference):
    return Plain.ToPlain(store)(reference.as_schema(), reference)


def plains(frames):
    """Frames given back, each symbol's reference in its plain form, a schema object as itself, a counted object, which
    the frame refuses, as "counted": what tells frames apart, a frame holding references made afresh in each transaction
    ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md))."""
    def one(frame, name):
        try:
            held = frame[name]
        except LookupError:
            return "counted"
        return plain(held) if store.name_of(held.as_schema()) == "References.OfObject" else held
    return [{name: one(frame, name) for name in frame} for frame in frames]


def path(singleton, *steps):
    """A path's plain form: from a singleton through entries, each (i, o, keys)."""
    nested = {"target": {}}
    for i, o, keys in reversed(steps):
        nested = {"entry": {"i": i, "o": o, "properties": keys, "next": nested}}
    return {"singleton": singleton, "next": nested}


def bound(held, references):
    """The frames of a map whose symbols are each given a reference: within the transaction that runs it, each symbol
    binds what its reference designates, so that the map gives one frame, or none where `held` holds nothing one
    designates ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md))."""
    return held.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols(dict(references))]))]))


def designated(held, reference):
    """What an object's reference designates in `held`, by a map whose one symbol is given it; None where it holds none."""
    frames = bound(held, {"x": reference})
    return frames[0]["x"] if frames else None


def framed(held, reference):
    """The frames of a map whose symbols are given the references a frame's reference binds, read off its plain form, as
    a program it is handed to reads them: the frame again, or none."""
    OfObject = held.registered("References.OfObject")
    binds = Plain.ToPlain(held)(reference.as_schema(), reference).get("binds", {})
    return bound(held, {name: Plain.FromPlain(held)(OfObject, given) for name, given in binds.items()})

<a id="REF-01"></a>
## REF-01 · Every store holds the reference schemas, its own; References.install, run again, writes nothing

In [ ]:
OfObject = store.registered("References.OfObject")
OfFrame, OfEntry = store.registered("References.OfFrame"), store.registered("References.OfEntry")
assert list(OfObject.properties) == ["path"] and OfObject.properties["path"].schema is store.registered("Paths.OfRoot")
OfProperties = store.registered("References.OfProperties")
assert list(OfFrame.properties) == ["binds", "variables", "assigns", "entries", "errors"]
assert OfFrame.properties["binds"].schema.element is OfObject and OfFrame.properties["assigns"].schema.element is OfProperties
assert OfFrame.properties["entries"].schema.element is OfEntry and list(OfProperties.properties) == ["object", "properties"]
assert list(OfEntry.properties) == ["relation", "links", "properties"]
assert OfEntry.properties["relation"].schema is store.registered("Paths.OfRoot")
assert OfEntry.properties["links"].schema.element is OfObject
OfError = store.registered("References.OfError")
assert OfFrame.properties["errors"].schema.element is OfError and list(OfError.properties) == ["subterm", "kind", "message"]
assert not any(schema.ref for schema in (OfObject, OfFrame, OfProperties, OfEntry, OfError))  # value objects, as a path is
other = B.Store()
assert other.registered("References.OfObject") is not OfObject  # each store's its own
names = {name: store.registered(name) for name in store.names()}
store.transaction(References.install)  # each found by its name
assert {name: store.registered(name) for name in store.names()} == names

<a id="REF-02"></a>
## REF-02 · An object's reference by its path: a singleton, an owned object, a schema object; each designated by a map whose symbol is given it

In [ ]:
assert plain(References.of(store, book)) == {"path": path("crm.TheBook")}
assert plain(References.of(store, ada)) == {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}
assert plain(References.of(store, Person)) == {"path": path(".", ("schemas", "", {"name": "crm"}),
                                                              ("schemas", "", {"name": "Person"}))}
for target in (book, ada):  # by a map whose symbol is given it: what it designates, by its reference again
    assert plain(designated(store, References.of(store, target))) == plain(target)
assert designated(store, References.of(store, Person)) is Person  # a schema object: itself
built = store.References.OfObject(lambda o: o.path(lambda p: p.singleton("crm.TheBook").next(lambda n: n.entry(
    lambda e: e.i("contacts").o("book").properties({"key": "bob"}).next(lambda n: n.target(lambda t: t))))))
assert plain(designated(store, built)) == plain(bob)  # written through the builders' fluent calls

<a id="REF-03"></a>
## REF-03 · A counted object has no reference, nor a frame that binds one: an upsert's, which the executor alone finds again

In [ ]:
made = store.transaction(notes)
assert plains(made) == [{"contact": plain(x), "note": "counted"} for x in (ada, bob, cy)]
with raises(LookupError, match="a counted 'crm.Note' has no reference: nothing names it"):
    made[0]["note"]  # the frame given back refuses it
with raises(LookupError, match="symbol 'note' of the frame: a counted 'crm.Note' has no reference"):
    References.of(store, made[0])
with raises(LookupError, match="a counted 'crm.Note' has no reference: nothing names it"), store.transaction():
    References.of(store, store.crm.Note(lambda n: n.text("hi").about([contact("ada")])))  # read within one: none either
again = store.transaction(notes)  # run again: the executor finds what it made, which nothing outside it names
assert plains(again) == plains(made) and not again[0].assigns  # its notes refused alike: nothing names them

<a id="REF-04"></a>
## REF-04 · A relation entry by its relation's path and all its links and properties: a keyed relation's, one without properties; refused where it links a counted object

In [ ]:
with store.transaction():
    held = next(iter(store.singleton("crm.TheBook").contacts))  # crm.Holds, keyed by `key`
    assert plain(References.of(store, held)) == {
        "relation": path(".", ("schemas", "", {"name": "crm"}), ("schemas", "", {"name": "Holds"})),
        "links": {"book": {"path": path("crm.TheBook")},
                  "contact": {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}},
        "properties": {"key": {"str": "ada"}}}
    knows = list(contact("ada").knows)
    assert plain(References.of(store, knows[1]))["properties"] == {"since": {"int": 2021}}
with store.transaction():
    store.crm.Person(contact("bob"), lambda b: b.knows(lambda a: a.insert(lambda e: e.known(contact("cy")))))  # since left unset
with store.transaction():
    bare = next(iter(contact("bob").knows))
    assert plain(References.of(store, bare))["properties"] == {}
with raises(LookupError, match="a counted 'crm.Note' has no reference"), store.transaction():  # crm.About links a note
    note = store.crm.Note(lambda n: n.text("hi").about([contact("ada")]))  # a note no frame gives back: made here
    References.of(store, next(iter(note.about)))


def known(x, y, *since):
    """The frames of a map over the references of `x` and `y`, whose spec is the crm.Knows entry linking them, with
    `since` where given: an entry's reference is matched by a pattern over the objects it links, one frame where the
    entry holds, none otherwise."""
    return store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols({"x": References.of(store, x), "y": References.of(store, y)})
        .spec([lambda _: _.x.knows([lambda e: e.known(_.y).since(*since) if since else e.known(_.y)])])]))]))


assert plains(known(ada, bob, 2020)) == [{"x": plain(ada), "y": plain(bob)}]
assert plains(known(ada, cy, 2021)) == [{"x": plain(ada), "y": plain(cy)}]
assert plains(known(bob, cy)) == [{"x": plain(bob), "y": plain(cy)}]  # since unset: the entry with its links alone
assert known(ada, cy, 1999) == [] and known(cy, ada) == []  # a property no entry has; links no entry has
held = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"b": References.of(store, book), "c": References.of(store, ada)})
    .spec([lambda _: _.b.contacts([lambda e: e.contact(_.c).key("ada")])])]))]))
assert plains(held) == [{"b": plain(book), "c": plain(ada)}]  # held

<a id="REF-05"></a>
## REF-05 · A frame a transaction gave back: each symbol's object; a map whose symbols are given its references gives the frame again

In [ ]:
frame = frames[0]
reference = References.of(store, frame)
assert sorted(plain(reference)["binds"]) == sorted(frame)
assert plain(reference)["binds"]["ada"] == plain(References.of(store, ada))
assert [a["properties"] for a in plain(reference)["assigns"]] == [{"name": {"str": n}} for n in ("Ada", "Bob", "Cy")]  # what it witnesses
assert plains(framed(store, reference)) == plains([frame])  # a map whose symbols are given the references it binds: the frame again
assert plain(References.of(store, {})) == {"binds": {}} and framed(store, References.of(store, {})) == [{}]
assert framed(store, store.References.OfFrame(lambda f: f)) == [{}]  # no binds: the empty frame

<a id="REF-06"></a>
## REF-06 · What is refused: a counted object nothing names, another store's object, what is not an object, an entry or a frame

In [ ]:
with store.transaction():
    loose = store.crm.Note(lambda n: n.text("by hand"))  # counted, written outright: nothing names it
    with raises(LookupError, match="a counted 'crm.Note' has no reference: nothing names it"):
        References.of(store, loose)
    with raises(LookupError, match="symbol 'loose' of the frame: a counted 'crm.Note' has no reference"):
        References.of(store, {"loose": loose})
    with raises(TypeError, match="the object belongs to another store"):
        References.of(other, contact("ada"))
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'x'"):
    References.of(store, "x")
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'x'"):
    References.of(store, {"x": "x"})  # a frame binding what is not an object
with store.transaction():
    store.crm.Person(contact("ada"), lambda a: a.trio(lambda t: t.insert(lambda e: e.b(contact("bob")).c(contact("cy")))))
with store.transaction(), raises(LookupError, match="has no path"):  # an entry of a relation with no name of its own
    References.of(store, next(iter(contact("ada").trio)))

<a id="REF-07"></a>
## REF-07 · A reference written and read back as a value; no frame where the store holds nothing it designates

In [ ]:
with store.transaction():  # as the store holds it now: an update of ada's rewrote her entries
    held = References.of(store, next(iter(store.singleton("crm.TheBook").contacts)))  # its reference, taken where read
backs = {}
for name, reference in (("ada", References.of(store, ada)), ("held", held), ("frame", References.of(store, frame))):
    text = JSON.ToJSON(store)(reference.as_schema(), reference)
    backs[name] = JSON.FromJSON(store)(reference.as_schema(), text)
    assert plain(backs[name]) == plain(reference)  # read back the same
assert plain(designated(store, backs["ada"])) == plain(ada) and plains(framed(store, backs["frame"])) == plains([frame])
empty = B.Store()  # its schemas and nothing else
empty.transaction(crm)
for target in (ada, Person):
    reference = References.of(store, target)
    back = JSON.FromJSON(empty)(empty.registered(store.name_of(reference.as_schema())),
                                JSON.ToJSON(store)(reference.as_schema(), reference))
    assert designated(empty, back) is (empty.registered("crm.Person") if target is Person else None)
nowhere = store.References.OfObject(lambda o: o.path(lambda p: p.singleton(".").next(lambda n: n.entry(
    lambda e: e.i("schemas").o("x").properties({"name": "crm"}).next(lambda n: n.target(lambda t: t))))))
assert designated(store, nowhere) is None  # a schema object's path whose steps name no entries

<a id="REF-08"></a>
## REF-08 · A frame as a witness: property values an object holds, and relation entries, each by its objects' references

In [ ]:
with store.transaction():  # their paths and an entry's reference, taken where they are read
    at_ada, at_bob = Paths.of(contact("ada")), Paths.of(contact("bob"))
    knowing = plain(References.of(store, next(iter(contact("ada").knows))))
witness = store.References.OfFrame(lambda f: f
    .binds({"ada": lambda o: o.path(at_ada), "bob": lambda o: o.path(at_bob)})
    .assigns([lambda x: x.as_object(lambda p: p.object(lambda o: o.path(at_ada)).properties({"name": "Ada"}))])
    .entries([lambda x: x.as_object(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))).links({
        "knower": lambda o: o.path(at_ada), "known": lambda o: o.path(at_bob)}).properties({"since": 2020}))]))
assert plain(witness)["assigns"] == [{"object": plain(References.of(store, ada)), "properties": {"name": {"str": "Ada"}}}]
assert plain(witness)["entries"] == [knowing]  # an OfEntry, as of() gives one
text = JSON.ToJSON(store)(witness.as_schema(), witness)
assert plain(JSON.FromJSON(store)(witness.as_schema(), text)) == plain(witness)  # carried, and read back the same
assert plains(framed(store, witness)) == [{"ada": plain(ada), "bob": plain(bob)}]  # its bindings, by a map over their references

<a id="REF-09"></a>
## REF-09 · A pattern's symbol given a reference in place of a schema: it binds what the reference designates, one frame, or none; its spec filters or is written as any symbol's, by the schema its path implies where the store holds nothing it designates; a symbol has exactly one of a schema and a reference

In [ ]:
reference = References.of(store, ada)  # a symbol given a reference binds what it designates, one frame each
known = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": reference, "y": Person}).spec([lambda _: _.x.knows([_.y])])]))]))
assert plains(known) == [{"x": plain(ada), "y": plain(bob)}, {"x": plain(ada), "y": plain(cy)}]  # from one object, along an adjacency
assert [plain(f["y"]) for f in store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": References.of(store, bob), "y": Person}).spec([lambda _: _.x.knows([_.y])])]))]))] == [plain(cy)]
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": reference}).spec([lambda _: _.x.name("Bob")])]))])) == []  # its spec filters, as any symbol's
upserted = store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"x": References.of(store, cy)}).spec([lambda _: _.x.name("Cy C.")])]))]))
assert plains(upserted) == [{"x": plain(cy)}] and upserted[0].assigns == {"x": {"name": "Cy C."}}  # an upsert writes its spec over what it binds
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"s": References.of(store, Person)}).spec([lambda _: _.s.name("crm.Person")])]))])) == [{"s": Person}]
nowhere = store.References.OfObject(lambda o: o.path(lambda p: p.singleton("crm.TheBook").next(lambda n: n.entry(
    lambda e: e.i("contacts").o("book").properties({"key": "zed"}).next(lambda n: n.target(lambda t: t))))))
for term in (lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": nowhere})])),
             lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols({"x": nowhere})]))):
    assert store.transaction(lambda t: t.mutations([term])) == []  # it designates nothing: no frame, nothing made
assert store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": nowhere}).spec([lambda _: _.x.name("Zed")])]))])) == []  # its spec, by the schema its path implies


def at(held, singleton, *steps):
    """A `References.OfObject` of `held`'s, from a singleton through entries, each (i, o, keys)."""
    def nested(n, rest):
        if not rest:
            return n.target(lambda t: t)
        (i, o, keys), *more = rest
        return n.entry(lambda e: e.i(i).o(o).properties(keys).next(lambda n: nested(n, more)))
    return held.References.OfObject(lambda o: o.path(lambda p: p.singleton(singleton).next(lambda n: nested(n, steps))))


client = B.Store()  # the schemas, not the data: its spec is written by the schemas its reference's path goes through
client.transaction(crm)
sent = client.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"x": at(client, "crm.TheBook", ("contacts", "book", {"key": "ada"})), "y": client.registered("crm.Person")})
    .spec([lambda _: _.x.knows([_.y])])]))]))
text = json.dumps(Plain.ToPlain(client).Transaction(sent))  # written as JSON, read where the data is, and run
assert plains(store.transaction(Plain.FromPlain(store).Transaction(json.loads(text)))) == plains(known)  # the frames, as given above


def firm(txn):
    """crm.Firm, owned through crm.Holds as a person is, by an adjacency also named `book`."""
    S = txn.Schemas
    return txn.uid("firm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Holds": S.OfRelation, "Firm": S.OfObject}).spec([
            lambda _: _.Holds.name("crm.Holds"),
            lambda _: _.Firm.name("crm.Firm").ref(True).adjacencies([
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True)]),
        ])]))])


firms = B.Store()
firms.transaction(crm)
firms.transaction(firm)
for held, given, why in (
        (client, at(client, "crm.Nope"), "starts from 'crm.Nope', which no schema declares a singleton"),
        (client, at(client, "crm.TheBook", ("nope", "book", {})),
         "steps through 'nope', an adjacency 'crm.Book' does not declare"),
        (client, at(client, "crm.TheBook", ("contacts", "nope", {})), "steps through 'contacts' of 'crm.Book', where not "
         "exactly one schema declares an adjacency 'nope' marked owner() on another link of its relation: found []"),
        (firms, at(firms, "crm.TheBook", ("contacts", "book", {"key": "ada"})), "steps through 'contacts' of 'crm.Book', "
         "where not exactly one schema declares an adjacency 'book' marked owner() on another link of its relation: "
         "found ['crm.Firm', 'crm.Person']"),
        (client, at(client, ".", ("schemas", "", {"name": "Nope"})), "names a schema object the store does not hold")):
    message = f"symbol 'x' is given a reference whose path {why}: a spec writes what its object's schema declares"
    with raises(LookupError, match=message), held.transaction():
        held.Schemas.OfPattern(lambda p: p.symbols({"x": given}).spec([lambda _: _.x.name("Zed")]))
with store.transaction():
    both = store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x").schema([Person]).reference(reference)]))
    neither = store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x")]))
    given = store.Schemas.OfPattern(lambda p: p.symbols({"x": reference}))
assert References.validate(both) == ["symbol 'x' has both a schema and a reference: a symbol has exactly one"]
assert References.validate(neither) == ["symbol 'x' has neither a schema nor a reference: a symbol has exactly one"]
assert References.validate(given) == [] and given.symbols["x"].schema is None
with raises(TypeError, match="References.validate takes a pattern, not 'x'"):
    References.validate("x")
with raises(TypeError, match="symbol 'x' has both a schema and a reference: a symbol has exactly one"), \
        store.transaction():
    store.Schemas.OfPattern(lambda p: p.symbols([lambda s: s.name("x").schema([Person]).reference(reference)]).spec([
        lambda _: _.x.name("Ada")]))

<a id="REF-10"></a>
## REF-10 · A frame annotated with errors: its reference carries each error's subterm, kind and message, written and read back as any value is

In [ ]:
with store.transaction():
    store.crm.Book(store.singleton("crm.TheBook"), lambda b: b.contacts({"eve": lambda e: e.name("Eve")}))
with store.transaction():  # written where Eve is read: Eve by her path
    eve = contact("eve")
    wire = Plain.ToPlain(store).Transaction(store.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m
        .symbols([lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.name("Ada")])]).then([lambda w: w.update(lambda u: u
        .apply([lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.knows([eve])])]))]))])))
knowing = Plain.FromPlain(store).Transaction(wire)
with store.transaction():
    store.crm.Book(store.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("eve"))))  # Eve deleted
try:
    store.transaction(knowing)
except LookupError as error:
    (annotated,) = error.frames
assert plains([annotated]) == [{"x": plain(ada)}] and len(annotated.errors) == 1
reference = References.of(store, annotated)
assert plain(reference) == {"binds": {"x": plain(References.of(store, ada))}, "assigns": [
    {"object": plain(References.of(store, ada)), "properties": {"name": {"str": "Ada"}}}], "errors": [{
    "subterm": "$.mutations[0].map.then[0].update", "kind": "LookupError",
    "message": "no object of the store has the path a spec refers to"}]}
text = JSON.ToJSON(store)(reference.as_schema(), reference)
assert plain(JSON.FromJSON(store)(reference.as_schema(), text)) == plain(reference)  # carried, the error itself not
assert "errors" not in plain(References.of(store, frame))  # a frame annotated with none: written as before
built = store.References.OfFrame(lambda f: f.binds({"x": lambda o: o.path(at_ada)}).errors([
    lambda x: x.as_object(lambda e: e.subterm("$.mutations[0].upsert").kind("ValueError").message("refused"))]))
assert plain(built)["errors"] == [{"subterm": "$.mutations[0].upsert", "kind": "ValueError", "message": "refused"}]

<a id="REF-11"></a>
## REF-11 · A parameter reference holds its parameter's name and nothing else: a value of the store's, written and read back, which no store resolves

In [ ]:
OfParameter = store.registered("References.OfParameter")
assert list(OfParameter.properties) == ["name"] and OfParameter.properties["name"].schema is store.registered("Schemas.OfStr")
assert not OfParameter.ref  # a value object, as the other references are
n = store.References.OfParameter(lambda p: p.name("n"))  # built by its name: References.of gives none
assert plain(n) == {"name": "n"}
text = JSON.ToJSON(store)(OfParameter, n)
assert text == '{"name": "n"}' and plain(JSON.FromJSON(store)(OfParameter, text)) == {"name": "n"}
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'n'"):
    References.of(store, "n")  # a parameter's name designates no object a store holds

<a id="REF-12"></a>
## REF-12 · A parameter reference stands in the place of anything with a schema: a property's scalar or value object, a container's element, a union's branch, an intersection's part and an entry's property; given as the value, or built through `as_parameter`; read through `as_parameter`, and as empty by every other visitor

In [ ]:
def par(txn):
    """par.Point, a value object; par.Shape, a union of scalars; par.Spot, a union of points; par.Series, a union of
    containers; par.Box, a singleton holding a value of each kind, and near boxes, at a gap."""
    S = txn.Schemas
    return txn.uid("par/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Point": S.OfObject, "Shape": S.OfUnion, "Spot": S.OfUnion, "Series": S.OfUnion, "Near": S.OfRelation,
                  "Box": S.OfObject})
        .spec([
            lambda _: _.Point.name("par.Point").properties({"x": S.OfInt, "y": S.OfInt}),
            lambda _: _.Shape.name("par.Shape").branches([
                lambda t: t.name("radius").schema([S.OfFloat]), lambda t: t.name("side").schema([S.OfInt])]),
            lambda _: _.Spot.name("par.Spot").branches([
                lambda t: t.name("corner").schema([_.Point]), lambda t: t.name("centre").schema([_.Point])]),
            lambda _: _.Series.name("par.Series").branches([
                lambda t: t.name("ints").schema([S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfInt]))]),
                lambda t: t.name("strs").schema([S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfStr]))])]),
            lambda _: _.Near.name("par.Near").links([lambda t: t.name("box"), lambda t: t.name("other")]).properties({
                "gap": S.OfInt}),
            lambda _: _.Box.name("par.Box").ref(True).singleton("par.TheBox").properties({
                "size": S.OfInt, "at": _.Point, "shape": _.Shape, "spot": _.Spot, "series": _.Series,
                "sizes": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfInt])),
                "points": S.OfContainer(lambda c: c.index([S.OfStr]).element([_.Point])),
                "pair": S.OfIntersection(lambda i: i.parts([
                    lambda t: t.name("a").schema([_.Point]), lambda t: t.name("b").schema([_.Point])])),
                "some": S.OfAny(lambda a: a.schemas([S.OfInt, _.Point])),
                "any": S.OfAny(lambda a: a),
                "param": txn.References.OfParameter,
            }).adjacencies([lambda t: t.name("near").relation([_.Near]).me("box"), lambda t: t.relation([_.Near]).me("other")]),
        ])]))])


held = B.Store()
held.transaction(par)
Box = held.registered("par.Box")
n = held.References.OfParameter(lambda p: p.name("n"))  # a value object, built by its parameter's name
with held.transaction():  # given as the value, in place of anything that has a schema
    box = held.singleton("par.TheBox")
    held.par.Box(box, lambda b: b.size(n).at(n).shape(lambda s: s.radius(n))
                 .spot(lambda s: s.corner(n)).series(lambda s: s.ints(n))
                 .sizes([1, n, 3]).points({"o": n}).pair(lambda p: p.a(lambda a: a.x(1)).b(n))
                 .some(n).any(n).param(n).near([lambda e: e.other(box).gap(n)]))


def parameter(read):
    """The name of the parameter a `Visitors.OfAny` holds a reference to, or None."""
    found = []
    read.as_parameter(lambda p: found.append(p.name.get()) or p)
    return found[0]


def of(prop):
    """A property's `Visitors.OfAny`."""
    found = []
    prop.as_value(lambda a: found.append(a) or a)
    return found[0]


def kinds(read):
    """What every other kind's visitor reads of a `Visitors.OfAny`: nothing, where a parameter reference stands in."""
    found = []
    read.as_scalar(lambda s: found.extend([] if s.get() is None else [s.get()]) or s)
    for select in ("as_object", "as_union", "as_intersection", "as_container"):
        getattr(read, select)(lambda v: found.extend([v] if v.as_schema() is not None else []) or v)
    return found


with held.transaction():
    box = held.singleton("par.TheBox")
    values = []
    for name in ("size", "at", "shape", "spot", "series", "sizes", "points", "pair", "some"):
        getattr(box, name).as_value(lambda a: values.append(a) or a)
    size, at, shape, spot, series, sizes, points, pair, some = values
    assert [parameter(a) for a in (size, at, some)] == ["n"] * 3 and [kinds(a) for a in (size, at, some)] == [[]] * 3
    assert box.size.get() is None and box.at.x.get() is None  # read as an unset value is
    assert [parameter(a) for a in (shape, spot, series, sizes, points, pair)] == [None] * 6  # each holds a value
    shape.as_union(lambda u: values.append(u.radius) or u)
    spot.as_union(lambda u: values.append(u.corner) or u)
    series.as_union(lambda u: values.append(u.ints) or u)
    pair.as_intersection(lambda i: values.append(i.b) or i)
    radius, corner, ints, b = values[-4:]
    assert radius.get() is None and corner.as_schema() is None and len(ints) == 0 and b.x.get() is None  # empty
    assert [parameter(m) for m in (radius, corner, ints, b)] == ["n"] * 4  # a branch's or a part's empty visitor holds it
    sizes.as_container(lambda c: values.append(c) or c)
    points.as_container(lambda c: values.append(c) or c)
    held_sizes, held_points = values[-2:]
    assert [parameter(held_sizes.at(i)) for i in (0, 1, 2)] == [None, "n", None] and kinds(held_sizes.at(1)) == []
    assert parameter(held_points.at("o")) == "n" and kinds(held_points.at("o")) == []
    assert box.near[0].gap.get() is None and parameter(of(box.near[0].gap)) == "n"  # an entry's property
    box.any.as_value(lambda a: values.append(a) or a)
    box.param.as_value(lambda a: values.append(a) or a)
    for admitted in values[-2:]:  # where its own schema is admitted, a parameter reference is a value of it, too
        assert parameter(admitted) == "n" and [o.name.get() for o in kinds(admitted)] == ["n"]
with held.transaction():  # built through as_parameter, where a callback writes a value
    box = held.singleton("par.TheBox")
    held.par.Box(box, lambda b: b.size(lambda s: s.as_value(lambda a: a.as_parameter(lambda p: p.name("m"))))
                 .at(lambda s: s.as_value(lambda a: a.as_parameter(lambda p: p.name("m")).as_parameter(lambda p: p.name("k"))))  # the same builder
                 .sizes(lambda s: s.as_value(lambda a: a.as_container(lambda c: c.at(0).as_parameter(n))))
                 .some(lambda s: s.as_value(lambda a: a.as_parameter(box.size))))  # a property's value, as held before
with held.transaction():
    box = held.singleton("par.TheBox")
    assert [parameter(of(getattr(box, name))) for name in ("size", "at", "some")] == ["m", "k", "n"]
    of(box.sizes).as_container(lambda c: values.append(parameter(c.at(0))) or c)
    assert values[-1] == "n"
    held.par.Box(box, lambda b: b.size(lambda s: s.as_value(lambda a: a.as_parameter(lambda p: p.name("j")))))  # edited
    assert parameter(of(box.size)) == "j"
    held.par.Box(box, lambda b: b.size(7).at(lambda a: a.x(2)).spot(lambda s: s.corner(lambda c: c.x(3))))
    assert box.size.get() == 7 and parameter(of(box.size)) is None and box.at.x.get() == 2  # a value written over it
    assert parameter(of(box.at)) is None
    values.append(box.spot.corner.x.get())
    assert values[-1] == 3

<a id="REF-13"></a>
## REF-13 · A parameter reference in the place of another schema's value is written marked, `{"$schema": "References.OfParameter", "name": n}`, and read back the same: in documents, snapshots, JSON, YAML, standalone values and a spec's payload; bare where its own schema is declared

In [ ]:
with held.transaction():
    box = held.singleton("par.TheBox")
    held.par.Box(box, lambda b: b.size(n).at(n).spot(lambda s: s.corner(n)))
marked = {"$schema": "References.OfParameter", "name": "n"}
with held.transaction():
    box = held.singleton("par.TheBox")
    document = Plain.ToPlain(held).OfObject(Box, box)
    written = {item["as_property"]["name"]: item["as_property"]["mutations"][0]["set"]["value"]
               for item in document["mutations"][0]["upsert"]["exists"][0]["spec"][0]["0"] if "as_property" in item}
assert [written[name] for name in ("size", "at", "some")] == [marked] * 3  # in another schema's place: marked
assert written["shape"] == {"radius": marked} and written["spot"] == {"corner": marked}
assert written["series"] == {"ints": marked} and written["pair"] == {"a": {"x": 1}, "b": marked}
assert written["sizes"] == [marked, marked, 3] and written["points"] == {"o": marked}
assert written["param"] == {"name": "n"}  # where its own schema is declared: bare
assert written["any"] == {"References.OfParameter": {"name": "n"}}  # of any schema: tagged by its schema, as any value is
again = B.Store()
again.transaction(par)
with held.transaction():
    snapshot = json.loads(json.dumps(Plain.ToPlain(held).Snapshot()))
Plain.FromPlain(again).Snapshot(snapshot)  # read back the same, in a snapshot
with again.transaction():
    assert Plain.ToPlain(again).OfObject(again.registered("par.Box"), again.singleton("par.TheBox")) == document
with held.transaction():
    for text in (JSON.ToJSON(held)(Box, held.singleton("par.TheBox")), YAML.ToYAML(held)(Box, held.singleton("par.TheBox"))):
        reader = JSON.FromJSON(again) if text.startswith("{") else YAML.FromYAML(again)
        with again.transaction():
            assert Plain.ToPlain(again).OfObject(again.registered("par.Box"), reader(again.registered("par.Box"), text)) == document
Point = held.registered("par.Point")
point = held.par.Point(lambda p: p.x(n).y(2))  # a standalone value
assert Plain.ToPlain(held)(Point, point) == {"x": marked, "y": 2}
assert Plain.ToPlain(held)(Point, Plain.FromPlain(held)(Point, {"x": marked, "y": 2})) == {"x": marked, "y": 2}
OfParameter = held.registered("References.OfParameter")
assert Plain.ToPlain(held)(OfParameter, Plain.FromPlain(held)(OfParameter, marked)) == {"name": "n"}  # its own schema's
sent = held.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"x": Box}).spec([lambda _: _.x.size(held.References.OfParameter(lambda p: p.name("w")))])]))]))
wire = Plain.ToPlain(held).Transaction(sent)  # a spec's payload, marked as a property's value is
assert wire["mutations"][0]["upsert"]["exists"][0]["spec"][0]["x"] == [{"as_property": {"name": "size", "mutations": [
    {"set": {"value": {"$schema": "References.OfParameter", "name": "w"}}}]}}]
held.transaction(Plain.FromPlain(held).Transaction(json.loads(json.dumps(wire))))
with held.transaction():
    assert parameter(of(held.singleton("par.TheBox").size)) == "w"

<a id="REF-14"></a>
## REF-14 · Validation accepts a parameter reference in any place: it is never a wrong value, and it holds an intersection's part; where its parameter context maps its name to nothing, it is partial, and valid

In [ ]:
with held.transaction():
    box = held.singleton("par.TheBox")
    held.par.Box(box, lambda b: b.size(n).at(n).shape(lambda s: s.radius(n))
                 .spot(lambda s: s.corner(n)).sizes([n]).points({"o": n}).some(n)
                 .pair(lambda p: p.a(lambda a: a.x(1)).b(lambda q: q.x(2)))
                 .near([lambda e: e.other(box).gap(n)]))
    assert Validators.Validate(held)(Box, box) == []  # never a wrong value, in any place
    outcome = Validators.Check(held)(Box, box)
    assert outcome.problems == [] and outcome.unknowns == [] and outcome.holds is True
    assert outcome.partials == [f"par.Box#0.{place}: parameter 'n' is neither assigned nor bound in its context" for place in (
        "size", "at", "shape.radius", "spot.corner", "series.ints", "sizes[0]", "points['o']", "some", "near[0].gap",
        "near[1].gap")] and outcome.result == "partial"  # an empty context maps no name: partial, and valid
    held.par.Box(box, lambda b: b.pair(lambda p: p.a(n).b(n)))
    assert Validators.Validate(held)(Box, box) == []  # an intersection's part that holds one is held
    point = held.par.Point(lambda p: p.x(1))  # a value object assigned: bound where a point is expected
    assert Validators.Validate(held)(Box, box, {"n": point}) == [
        f"par.Box#0.{place}: parameter 'n' is assigned a value of another schema than its place expects"
        for place in ("size", "shape.radius", "series.ints", "sizes[0]", "near[0].gap", "near[1].gap")]

<a id="REF-15"></a>
## REF-15 · A place that holds no parameter reference reads as empty through `as_parameter`; a read-only visitor refuses a write, a builder refuses a value that is not a parameter reference, and a schema object's value holds none

In [ ]:
with held.transaction():
    box = held.singleton("par.TheBox")
    held.par.Box(box, lambda b: b.size(3).shape(lambda s: s.radius(n)))
with held.transaction():
    box = held.singleton("par.TheBox")
    assert parameter(of(box.size)) is None  # a value of its own schema: no parameter reference held
    with raises(ReadOnlyError, match="property 'size' holds no parameter reference"):
        of(box.size).as_parameter(n)  # read-only: refused, saying why
    with raises(ReadOnlyError, match="property 'at' is read-only; write through a builder"):
        of(box.at).as_parameter(n)
    with raises(ReadOnlyError, match="branch 'radius' holds a parameter reference"):
        box.shape.radius.as_parameter(n)
    with raises(TypeError, match="property 'size' takes a parameter reference or a Spec, got int"):
        held.par.Box(box, lambda b: b.size(lambda s: s.as_value(lambda a: a.as_parameter(4))))
scalar = B.OfScalar.Builder(None, True, held).name("par.Name")  # a schema object's value holds none
reasons = []
scalar.name.as_value(lambda v: v.as_parameter(lambda p: reasons.append(p.as_schema()) or p))
assert reasons == [None]
with raises(TypeError, match="property 'name' holds no parameter reference"):
    scalar.name.as_value(lambda v: v.as_parameter(n))

<a id="REF-16"></a>
## REF-16 · A parameter reference stands in the place of an object a link links: given as the object is, or by the peer shorthand; read through the link as an empty visitor whose `as_parameter` hands it over; holding no entry table, keyed by its parameter's name, never reached, accepted by validation, halting comparison, removed by its name, and refused by a relation that owns what it links

In [ ]:
def lnk(txn):
    """lnk.Hub, a singleton owning its nodes by key; lnk.Node, each linked to the next, at a weight."""
    S = txn.Schemas
    return txn.uid("lnk/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Holds": S.OfRelation, "Next": S.OfRelation, "Tags": S.OfRelation, "Hub": S.OfObject, "Node": S.OfObject})
        .spec([
            lambda _: _.Holds.name("lnk.Holds").links([lambda t: t.name("hub"), lambda t: t.name("node")]).properties({
                "key": S.OfStr}).uniques([["hub", "key"], ["node"]]),
            lambda _: _.Next.name("lnk.Next").links([lambda t: t.name("from"), lambda t: t.name("to")]).properties({
                "w": S.OfInt}).uniques([[]]),  # a set
            lambda _: _.Tags.name("lnk.Tags").links([lambda t: t.name("tagger"), lambda t: t.name("tagged"),
                                                     lambda t: t.name("tag")]),
            lambda _: _.Hub.name("lnk.Hub").ref(True).singleton("lnk.TheHub").adjacencies({
                "nodes": lambda t: t.relation([_.Holds]).me("hub")}),
            lambda _: _.Node.name("lnk.Node").ref(True).properties({"label": S.OfStr}).adjacencies([
                lambda t: t.name("hub").relation([_.Holds]).me("node").owner(True),
                lambda t: t.name("next").relation([_.Next]).me("from"),
                lambda t: t.name("prev").relation([_.Next]).me("to"),
                lambda t: t.name("tags").relation([_.Tags]).me("tagger"),
                lambda t: t.relation([_.Tags]).me("tagged"),  # anonymous
            ]),
        ])]))])


graph = B.Store()
graph.transaction(lnk)
Hub, Node = graph.registered("lnk.Hub"), graph.registered("lnk.Node")
k = graph.References.OfParameter(lambda p: p.name("k"))
with graph.transaction():  # in the place of the object a link links: given as the object is, or by the peer shorthand
    hub = graph.singleton("lnk.TheHub")
    graph.lnk.Hub(hub, lambda h: h.nodes({"a": lambda e: e.node(lambda x: x.label("A")),
                                           "b": lambda e: e.node(lambda x: x.label("B"))}))
    a, b = hub.nodes({"key": "a"}), hub.nodes({"key": "b"})
    graph.lnk.Node(a, lambda x: x.next([lambda e: e.to(k).w(1), lambda e: e.to(b).w(2), lambda e: e.to(k).w(1)]))
    graph.lnk.Node(b, lambda x: x.next([k]))  # the peer shorthand
with graph.transaction():
    hub = graph.singleton("lnk.TheHub")
    a, b = hub.nodes({"key": "a"}), hub.nodes({"key": "b"})
    assert len(a.next) == 2 and len(b.next) == 1  # an equal entry elided: one parameter, one key, by its name
    entry = a.next[0]
    assert entry.to.as_schema() is None and entry.to.label.get() is None and list(entry.to.next) == []  # empty
    names = []
    entry.to.as_parameter(lambda p: names.append(p.name.get()) or p)
    b.next[0].to.as_parameter(lambda p: names.append(p.name.get()) or p)
    assert names == ["k", "k"] and a.next[1].to is b and entry.w.get() == 1
    assert list(b.prev) == [a.next[1]]  # an object linked holds its entry; a parameter reference holds none
    assert Reachable.of(hub) == [hub, a, b]  # never reached
    assert Validators.Validate(graph).Reachable(Hub, hub) == []  # accepted in an object's place: partial, and valid
    assert Validators.Check(graph).Reachable(Hub, hub).result == "partial"
    assert C.OfEntry(Node.adjacencies["next"], entry).compare(C.OfEntry(Node.adjacencies["next"], entry)) is None
    assert C.OfEntry(Node.adjacencies["next"], a.next[1]).compare(C.OfEntry(Node.adjacencies["next"], a.next[1])) == 0
    reference = References.of(graph, entry)  # an entry's reference: the parameter reference in the object's place
    assert Plain.ToPlain(graph)(reference.as_schema(), reference)["links"]["to"] == {
        "$schema": "References.OfParameter", "name": "k"}
    graph.lnk.Node(b, lambda x: x.next(lambda adj: adj.remove(lambda e: e.to(k))))  # removed by its parameter's name
    assert len(b.next) == 0
    graph.lnk.Node(b, lambda x: x.next([k]))
    built = []
    graph.lnk.Node(b, lambda x: x.next(lambda adj: adj.insert(lambda e: built.append(e.to(entry.to).w(9).to.as_schema()))))
    assert built == [None] and b.next[1].to.as_schema() is None  # relinked to what another entry links; read as empty
    graph.lnk.Node(b, lambda x: x.next({4: k}))  # a keyed mapping's value: the link takes it
    graph.lnk.Node(b, lambda x: x.next(lambda adj: adj.remove(lambda e: e.to(a)).insert(lambda e: e.to(k).w(7))))
    assert [e.w.get() for e in b.next] == [None, 9, 4, 7]  # checked against its unique clauses where a removal missed
    with raises(TypeError, match="link 'to' holds a parameter reference, which has no members to write"):
        graph.lnk.Node(b, lambda x: x.next([lambda t: t.to(k).label("K")]))
    with raises(ValueError, match="link 'node' of an entry of 'lnk.Holds' holds a parameter reference, which a relation "
                                  "that owns what it links takes in no link"):
        graph.lnk.Hub(hub, lambda h: h.nodes({"c": k}))

<a id="REF-17"></a>
## REF-17 · A link to a parameter reference is set as `{"reference": {"$schema": "References.OfParameter", "name": n}}`, beside a symbol or a path, and read back the same: in documents, snapshots, a spec and its effect, matched by its parameter's name, and in an expression's nested form; a malformed one refused

In [ ]:
marked = {"$schema": "References.OfParameter", "name": "k"}
with graph.transaction():
    hub = graph.singleton("lnk.TheHub")
    a = hub.nodes({"key": "a"})
    document = Plain.ToPlain(graph).OfObject(Node, a)
    at_a = References.of(graph, a)
    assert {"as_link": {"name": "to", "mutations": [{"set": {"reference": marked}}]}} in [
        part for item in document["mutations"][0]["upsert"]["exists"][0]["spec"] for members in item.values()
        for member in members if "as_adjacency" in member for verb in member["as_adjacency"]["mutations"]
        for entry in verb["insert"]["entries"] for part in entry["mutations"]]  # written marked beside a symbol or a path
    snapshot = json.loads(json.dumps(Plain.ToPlain(graph).Snapshot()))
copy = B.Store()
copy.transaction(lnk)
Plain.FromPlain(copy).Snapshot(snapshot)  # read back the same, in a snapshot
with copy.transaction():
    held = copy.singleton("lnk.TheHub").nodes({"key": "a"})
    Copied = copy.registered("lnk.Node")
    assert Plain.ToPlain(copy).OfObject(Copied, held) == document
    read = JSON.FromJSON(copy)(Copied, json.dumps(document))  # and in a document, read where its object is
    assert Plain.ToPlain(copy).OfObject(Copied, read) == document
seen = []
end = Changes.observe(graph, lambda journal: seen.append(Plain.ToPlain(graph)(graph.registered("Changes.Journal"), journal)))
spec = graph.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"x": at_a}).spec([
        lambda _: _.x.next([lambda e: e.to(graph.References.OfParameter(lambda q: q.name("j"))).w(3)])])]))]))
wire = json.loads(json.dumps(Plain.ToPlain(graph).Transaction(spec)))  # a spec's link set, marked
assert wire["mutations"][0]["upsert"]["exists"][0]["spec"][0]["x"][0]["as_adjacency"]["mutations"][0]["insert"][
    "entries"][0]["mutations"][0] == {"as_link": {"name": "to", "mutations": [
        {"set": {"reference": {"$schema": "References.OfParameter", "name": "j"}}}]}}
graph.transaction(Plain.FromPlain(graph).Transaction(wire))  # run: it links the parameter reference
graph.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"x": at_a}).spec([lambda _: _.x.next({5: graph.References.OfParameter(lambda q: q.name("i"))})])]))]))
with graph.transaction():  # removed: its journal removes it by its parameter's name
    graph.lnk.Node(graph.singleton("lnk.TheHub").nodes({"key": "a"}), lambda x: x.next(lambda adj: adj.remove(
        lambda e: e.to(graph.References.OfParameter(lambda q: q.name("i"))))))
end()
assert [change["entry"]["verb"] for (transaction,) in seen for change in transaction["changes"]] == [
    "insert", "insert", "remove"]
assert seen[0][0]["changes"][0]["entry"]["links"]["to"] == {"reference": {"$schema": "References.OfParameter", "name": "j"}}
with graph.transaction():
    hub = graph.singleton("lnk.TheHub")
    held, names = list(hub.nodes({"key": "a"}).next), []
    for entry in held:
        if entry.to.as_schema() is None:
            entry.to.as_parameter(lambda p: names.append(p.name.get()) or p)
    assert names == ["k", "j"] and held[1].to is hub.nodes({"key": "b"})  # linked, in its transaction


def linking(name):
    """The frames of a map whose node's spec gives an entry linking a parameter reference to `name`."""
    return graph.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols({"x": Node}).spec([lambda _: _.x.next([lambda e: e.to(graph.References.OfParameter(
            lambda q: q.name(name)))])])]))]))


assert len(linking("j")) == 1 and len(linking("k")) == 2 and linking("z") == []  # matched by its parameter's name
with raises(DecodeError, match="set.reference.name: expected str, got int"):  # never guessed
    bad = json.loads(json.dumps(wire))
    bad["mutations"][0]["upsert"]["exists"][0]["spec"][0]["x"][0]["as_adjacency"]["mutations"][0]["insert"]["entries"][0][
        "mutations"][0]["as_link"]["mutations"][0]["set"]["reference"]["name"] = 1
    Plain.FromPlain(graph).Transaction(bad)
expression = graph.Mutations.OfTransaction(lambda t: t.mutations([k]))  # in the place of a term an expression links
nested = Plain.ToPlain(graph).Transaction(expression)
assert nested == {"mutations": [marked]}
assert Plain.ToPlain(graph).Transaction(Plain.FromPlain(graph).Transaction(nested)) == nested
below = B.Store()  # a store below: an entry another writes there, which no object present above has an adjacency to
below.transaction(lnk)  # by name, is applied to those present directly, a parameter reference as itself
with below.transaction():
    graph_hub = below.singleton("lnk.TheHub")
    below.lnk.Hub(graph_hub, lambda h: h.nodes({"a": lambda e: e.node(lambda x: x.label("A"))}))
above = B.Store(below)
above.transaction(lnk)
with above.transaction():
    above.singleton("lnk.TheHub").nodes({"key": "a"})  # present above
with below.transaction():
    hub = below.singleton("lnk.TheHub")
    below.lnk.Hub(hub, lambda h: h.nodes({"c": lambda e: e.node(lambda x: x.tags(lambda adj: adj.insert(
        lambda t: t.tagged(hub.nodes({"key": "a"})).tag(below.References.OfParameter(lambda q: q.name("t"))))))}))
with above.transaction():
    (tagging,) = Proxies._linking(above.registered("lnk.Tags"), "tagged", above.singleton("lnk.TheHub").nodes({"key": "a"}))
    tag = []
    tagging.tag.as_parameter(lambda p: tag.append(p.name.get()) or p)
    assert tag == ["t"]